In [3]:
import numpy as np
import matplotlib.pyplot as plt
import copy

import pyccl as ccl

from scipy.special import jv
from scipy.integrate import romb

import treecorr
import healpy as hp

from heavy_halo_filter import build_contamination_map, flag_contaminated_sources

In [10]:
# Load in source distribution from flagship

from astropy.io import fits
import numpy as np

# fits_file = fits.util.get_testdata_filepath("23586.fits")
hdul = fits.open("/home/milan/Desktop/thesis/flagship_sim/25224.fits")
hdr = hdul[1].header
data = hdul[1].data

z = data['true_redshift_gal']
flux = data['euclid_vis']
mag = -2.5 * np.log10(flux) - 48.6


# create a mask in redshift:
z_min = 0.5
z_max = 2
mask = np.where((z > z_min) & (z < z_max) & (mag < 25))

z = z[mask]
mag = mag[mask]
ra = data['ra_gal'][mask]
dec = data['dec_gal'][mask]

kappa = data['kappa'][mask]
gamma1 = data['gamma1'][mask]
gamma2 = data['gamma2'][mask]
eps1 = data['eps1_gal'][mask]
eps2 = data['eps2_gal'][mask]

u_flux_abs = data['cfht_u_abs'][mask]
r_flux_abs = data['subaru_r_abs'][mask]

z_edges = np.linspace(0.5, 2, 31)


In [11]:
for mag_threshold in [20, 21, 22, 23, 24]:

    contamination_map = build_contamination_map(
        ra, dec, z, mag_bright=mag, mag_threshold=mag_threshold, theta_arcmin=0.5, nside=2*4096
    )

    contaminated_sources = flag_contaminated_sources(
        ra, dec, z, contamination_map
    )
    print(f"fraction of contaminated sources for mag_threshold {mag_threshold}:", 
          np.sum(contaminated_sources)/len(contaminated_sources))


fraction of contaminated sources for mag_threshold 20: 0.08027961128842413
fraction of contaminated sources for mag_threshold 21: 0.45057445871674917
fraction of contaminated sources for mag_threshold 22: 0.8314789696174042
fraction of contaminated sources for mag_threshold 23: 0.94244549419157
fraction of contaminated sources for mag_threshold 24: 0.9701201410419655


In [12]:
for theta_arcmin in [0.2, 0.5, 1, 2]:

    mag_threshold = 22 # this is apparent magnitude but it should be absolute magnitude

    contamination_map = build_contamination_map(
        ra, dec, z, mag_bright=mag, mag_threshold=mag_threshold, theta_arcmin=theta_arcmin, nside=2*4096
    )

    contaminated_sources = flag_contaminated_sources(
        ra, dec, z, contamination_map
    )
    print(f"fraction of contaminated sources for theta_threshold {theta_arcmin} arcmin:", 
          np.sum(contaminated_sources)/len(contaminated_sources))

fraction of contaminated sources for theta_threshold 0.2 arcmin: 0.5706559140493221
fraction of contaminated sources for theta_threshold 0.5 arcmin: 0.8314789696174042
fraction of contaminated sources for theta_threshold 1 arcmin: 0.953648490547969
fraction of contaminated sources for theta_threshold 2 arcmin: 0.9865755774861839


In [6]:
# now lets make cuts so we only select blue galaxies, for this we follow along 
# Euclid preparation:
# Calibrated intrinsic galaxy alignments in the Euclid Flagship simulation
# Euclid Collaboration: K. Hoffmann ....
# page 4 bottom right

h = 0.67
u_mag_abs = -2.5 * np.log10(u_flux_abs) - 48.6 + 5 * np.log10(h)
r_mag_abs = -2.5 * np.log10(r_flux_abs) - 48.6 + 5 * np.log10(h)

mask = (u_mag_abs - r_mag_abs < 1.32)

fraction_blue_galaxies = np.sum(mask) / len(mask)
print("fraction of blue galaxies: ", fraction_blue_galaxies)

z = z[mask]
ra = ra[mask]
dec = dec[mask]

kappa = kappa[mask]
gamma1 = gamma1[mask]
gamma2 = gamma2[mask]
eps1 = eps1[mask]
eps2 = eps2[mask]

fraction of blue galaxies:  0.7365159467501206
